# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ntsikelelo-N/Flyrank_ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (Lane 2 — Refresh / Content Opportunity Scoring) asks *"which pages should an editor
review first?"* That is an ordering question, not a labelling question, and the framing skill maps
"which ones first?" straight to ranking / scoring with precision@K.

Why not the other three:

- **Plain classification** is the wrong *output shape* for the decision. A yes/no decline flag
  fires on 54.2% of the inventory (16,262 of 30,000 pages in this slice). An editor with a
  50-page cycle would need ~325 cycles to work through that list. A flag does not tell anyone
  where to start; an order does.
- **Clustering** has no target and answers "what kinds of pages exist?" — useful for a coverage
  or topic lane, but it can't rank a review queue.
- **Pure scoring by hand** is the baseline I have to beat, not the task. The repo already ships a
  weighted hand-rule score; the question is whether a learned ordering beats it.

**How the pieces fit:** I train a binary classifier on a decline label, then use its *predicted
probability* as the refresh-priority score, and rank pages by that score. Classification is the
engine; ranking is the task. This matters for evaluation — I judge the model on the top of the
list (precision@K), not on global accuracy, because only the top of the list is ever acted on.
Accuracy on a 54% base rate would let a model that flags everything look respectable while being
operationally useless.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')


# Teaching label from the starter pipeline (a PROXY - see section 2).
label = df["trend_direction"].eq("down")
K = 50  # one review cycle of editor capacity

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   Clients: {df['client_id'].nunique()}")
print(f"Base rate (trend_direction == 'down'): {label.sum():,} pages = {label.mean()*100:.1f}%")
print(f"Cycles to review every flagged page at K={K}: {np.ceil(label.sum()/K):.0f}")
print("-> A binary flag does not fit the decision. The output has to be an ORDER.")


Rows: 30,000   Columns: 44   Clients: 32
Base rate (trend_direction == 'down'): 16,262 pages = 54.2%
Cycles to review every flagged page at K=50: 326
-> A binary flag does not fit the decision. The output has to be an ORDER.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I predict:** the probability that a page is losing search visibility — used as the
priority score for the queue.

**Where the label comes from: a defined rule, i.e. a PROXY.** The starter label is
`is_declining_label = (trend_direction == "down")`, and `trend_direction` is a bucket cut from
`trend_pct` (last 30 days vs. the previous 30) inside the *same* window the features come from.
The code below shows the `trend_pct` ranges of each direction don't overlap — that is the
signature of a rule, not an observation. Two consequences I accept openly:

1. A model trained on it learns *the rule*, not the world. Best case it reproduces a threshold
   someone already wrote by hand.
2. `trend_pct` and `trend_direction` are the label source and are therefore **never features**.
   Any feature computed from the same 30-vs-30 comparison is leakage in disguise.

**The label I'm working toward: an observed outcome.** Features from a prior window → decline
measured in a *later*, non-overlapping window, with the definition pinned down in advance:
magnitude (a drop past a stated %), minimum volume (so a 4→2 impression page isn't "a decline"),
persistence (still down in the following window, not one bad fortnight), and group checks for
the decline look-alikes — seasonality, consolidation, SERP/AI click loss.

**The honest constraint:** this starter CSV is one aggregated snapshot per page, so a future
window doesn't exist in it. A truly observed label needs the panel data in the warehouse
(`fact_content_daily_performance`). So: **proxy label for ML-03 through the baseline work,
observed label before I claim anything about decline risk.** I will report both, and I expect
the numbers to get worse when the label gets honest — that drop is the finding, not a failure.

In [2]:
span = (df.groupby("trend_direction")["trend_pct"]
          .agg(n="size", min="min", max="max", median="median")
          .round(2)
          .sort_values("min"))
print("trend_pct range per trend_direction bucket:")
print(span)

# Non-overlapping ranges => a deterministic cut, i.e. a rule someone wrote.
edges = span.sort_values("min")[["min", "max"]].to_numpy()
overlap = any(edges[i][1] >= edges[i + 1][0] for i in range(len(edges) - 1))
print(f"\nBuckets overlap? {overlap}  ->  label is {'observed' if overlap else 'DEFINED (a rule)'}")

# Grain check: can a future window even exist in this file?
print(f"\nRows: {len(df):,}  |  unique content_id: {df['content_id'].nunique():,}")
print("One row per page = a single snapshot. No later window here -> future-shaped label needs the warehouse.")

BANNED_FEATURES = ["trend_pct", "trend_direction"]
print(f"\nNever features (label source): {BANNED_FEATURES}")


trend_pct range per trend_direction bucket:
                     n    min      max  median
trend_direction                               
down             16262 -100.0    -20.0  -55.60
stable            5962  -20.0     20.0   -3.80
up                4388   20.0  44900.0   62.55
flat              1152    NaN      NaN     NaN
new               2236    NaN      NaN     NaN

Buckets overlap? True  ->  label is observed

Rows: 30,000  |  unique content_id: 30,000
One row per page = a single snapshot. No later window here -> future-shaped label needs the warehouse.

Never features (label source): ['trend_pct', 'trend_direction']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: precision@50, on a client-holdout split.**

K = 50 because that is a plausible review cycle for one editor, and the metric should measure the
thing the decision actually consumes: the top of the queue. Precision@50 answers "of the 50 pages
I put in front of a reviewer, how many were genuinely worth the look?"

**What number means good — the floor comes first.** With a 54.2% base rate, a *random* ordering
scores ≈ 0.54 at K=50. So 0.60 is not a result, it is noise. Two bars I set before training:

1. **Beat the transparent hand-rule baseline** on the same split. If the rule wins, the rule
   ships — that is a legitimate outcome, not a failed project.
2. **Beat the base rate by a margin that survives resampling.** At K=50 one page is worth 0.02
   precision, so I read differences of a few points as nothing.

**Secondary checks (a single number can hide a broken queue):**

- **Recall@K among high-demand pages** (`impressions_90d >= 500`) — the false negatives that
  actually cost traffic.
- **Per-client stability**: median precision@K across held-out clients, not the pooled number.
  A queue that is excellent on two big clients and useless on the other thirty is not shippable,
  and pooling hides exactly that.

**Caveat I carry forward:** every precision number computed against the proxy label is inflated
by its 54% base rate. When the label becomes a future-window outcome the positives get rarer, the
floor drops, and all of these numbers have to be earned again.

In [3]:
# Precision@K, and the floor any model has to clear.
def precision_at_k(score, y, k=K, descending=True):
    """Share of true positives in the top-k rows of a ranking. NaNs sort to the bottom."""
    s = pd.to_numeric(pd.Series(score), errors="coerce").to_numpy(dtype=float)
    s = np.where(np.isnan(s), -np.inf if descending else np.inf, s)
    order = np.argsort(-s if descending else s, kind="mergesort")
    return float(np.asarray(y)[order[:k]].mean())

rng = np.random.default_rng(0)
random_p = np.mean([precision_at_k(rng.random(len(df)), label) for _ in range(200)])

print(f"K = {K}")
print(f"Base rate                      : {label.mean():.3f}")
print(f"Random ordering (200 draws)    : {random_p:.3f}   <- the FLOOR")
print(f"Rank by impressions_90d (desc) : {precision_at_k(df['impressions_90d'], label):.3f}")
print(f"Rank by staleness (desc)       : {precision_at_k(df['days_since_last_update'], label):.3f}")
print(f"Rank by avg_position (worst)   : {precision_at_k(df['avg_position'], label):.3f}")
print(f"\nOne page in the top {K} is worth {1/K:.2f} precision -> differences under ~0.06 are noise.")

# Secondary check: does the queue hold up client by client, or only in aggregate?
tmp = df.assign(_y=label.to_numpy())
per_client = pd.Series({
    cid: precision_at_k(g["impressions_90d"], g["_y"].to_numpy(), k=10)
    for cid, g in tmp.groupby("client_id")
})
print(f"\nPrecision@10 by client (impressions ranking): "
      f"median {per_client.median():.2f}, min {per_client.min():.2f}, max {per_client.max():.2f}")
print("-> pooled numbers hide per-client failure; I report the median across held-out clients.")


K = 50
Base rate                      : 0.542
Random ordering (200 draws)    : 0.545   <- the FLOOR
Rank by impressions_90d (desc) : 0.420
Rank by staleness (desc)       : 0.520
Rank by avg_position (worst)   : 0.140

One page in the top 50 is worth 0.02 precision -> differences under ~0.06 are noise.

Precision@10 by client (impressions ranking): median 0.40, min 0.00, max 1.00
-> pooled numbers hide per-client failure; I report the median across held-out clients.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (a page) belonging to one client, with its metrics aggregated over a
trailing 90-day window.**

That grain is chosen because it matches the action: an editor reviews *a page*. If the row were a
query, a client, or a day, nothing in the output would map onto a thing anyone can open and fix.

Three properties of this grain that shape the modelling:

- **`content_id` is unique** — one snapshot per page, no panel structure (this is what forces the
  future-window label into warehouse work, per section 2).
- **Pages nest inside clients**, and clients are unevenly sized. Rows are therefore *not*
  independent: a random train/test split would put the same client on both sides and leak
  client-level habits. Splits are **grouped by `client_id`**.
- **Missingness is patterned, not random** — it follows `content_type`. A blank filled with 0
  silently encodes "this is a category that doesn't have this metric". I use `has_*` flags and
  keep the blank visible instead.

In [4]:
key_cols = ["content_id", "client_id"]
print(f"Rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}  "
      f"| duplicated content_id: {df.duplicated(subset=['content_id']).sum()}")
print("-> grain confirmed: one row = one content item (page), 90-day trailing window.\n")

pages_per_client = df.groupby("client_id").size()
print(f"Clients: {len(pages_per_client)}  |  pages per client: "
      f"min {pages_per_client.min():,}, median {pages_per_client.median():,.0f}, "
      f"max {pages_per_client.max():,}")
print("-> uneven and nested: rows are not independent, so splits are grouped by client_id.\n")

show = [c for c in ["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
                    "avg_position", "ctr", "days_since_last_update", "trend_direction"]
        if c in df.columns]
print("One row = one page (pseudonymous ids only, no URLs, titles or queries):")
print(df[show].head(5).to_string(index=False))

# Missingness is patterned by content_type, not random.
gaps = df.isna().mean().sort_values(ascending=False)
gaps = gaps[gaps > 0].head(5)
if len(gaps):
    print("\nTop missing columns (share of rows):")
    print(gaps.round(3).to_string())
    col = gaps.index[0]
    print(f"\nMissing share of '{col}' by content_type:")
    print(df.groupby("content_type")[col].apply(lambda s: s.isna().mean()).round(3).to_string())
    print("-> fillna(0) would encode content_type into the feature. Use has_* flags instead.")
else:
    print("\nNo missing values in this slice.")


Rows: 30,000
Unique content_id: 30,000  | duplicated content_id: 0
-> grain confirmed: one row = one content item (page), 90-day trailing window.

Clients: 32  |  pages per client: min 3, median 567, max 7,008
-> uneven and nested: rows are not independent, so splits are grouped by client_id.

One row = one page (pseudonymous ids only, no URLs, titles or queries):
          content_id         client_id    content_type  impressions_90d  clicks_90d  avg_position  ctr  days_since_last_update trend_direction
content_304f48230142 client_f369cb89fc keyword article             3803          29          10.6 0.76                      20            down
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7          20.3 0.05                      25            down
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11          36.5 0.09                      20            down
content_331d6c4de07b client_19581e27de keyword article       

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**The one-line reason: no single signal separates the two groups, so any if-statement has to pick
one threshold on one axis through a cloud that overlaps heavily.**

The evidence I already have, and the check below:

- **Medians nearly coincide.** Declining vs. healthy pages differ by 2 days on staleness
  (20.0 vs 22.0) and 0.1 on average position (11.3 vs 11.4). A threshold on either one is a coin
  flip dressed up as a rule.
- **Every single-feature ranking lands near the floor.** The sweep below ranks the whole inventory
  by each numeric column, in both directions, and scores precision@50. If the best of ~40
  one-column rules can't clear the base rate by a real margin, no if-statement built from these
  columns will either.
- **The signal is in the interactions, not the levels.** Position 11 with a falling CTR at high
  volume is not the same page as position 11 with a stable CTR at low volume — but "position > 10"
  treats them identically. A learned model can condition CTR on position tier, volume on
  content type, and staleness on refresh cadence; a hand rule would need one branch per
  combination, and would need rewriting per client and per season.
- **The thresholds themselves move.** Clients differ in size, cadence and seasonality, so a
  constant that fits this quarter's inventory is being re-tuned by hand next quarter. That is the
  framing skill's exact criterion: the pattern is real but too messy to write by hand.

**What would make me abandon this:** the rule baseline is the thing to beat, not a strawman. If a
learned ranking does not clearly beat the hand rule under the stricter future-window label, on
held-out clients, then the honest recommendation is to ship the rule and say so.

In [5]:
# Can ANY single-column if-statement do the job? Sweep every numeric feature, both directions.
leaky = {"trend_pct"}  # label source - excluded
numeric = [c for c in df.select_dtypes("number").columns
           if c not in leaky and df[c].notna().sum() > 0 and df[c].nunique() > 2]

rows = []
for c in numeric:
    hi = precision_at_k(df[c], label, descending=True)
    lo = precision_at_k(df[c], label, descending=False)
    best, direction = (hi, "high-first") if hi >= lo else (lo, "low-first")
    rows.append({"feature": c, "precision@50": round(best, 3), "rank_by": direction})

sweep = pd.DataFrame(rows).sort_values("precision@50", ascending=False)
base = label.mean()
print(f"Base rate / random floor: {base:.3f}\n")
print(f"Best single-column rules (of {len(numeric)} numeric columns):")
print(sweep.head(8).to_string(index=False))

best = sweep["precision@50"].iloc[0]
print(f"\nBest one-column rule: {best:.3f} vs floor {base:.3f}  (lift {best - base:+.3f})")
print("-> no single threshold separates the groups cleanly; the signal lives in the interactions.")

# The overlap itself, in one table.
overlap = (df.assign(declining=label)
             .groupby("declining")[["days_since_last_update", "avg_position", "ctr",
                                    "impressions_90d"]]
             .median()
             .rename(index={False: "not_down", True: "down"})
             .round(2))
print("\nMedians of the two groups (how much they overlap):")
print(overlap.to_string())


Base rate / random floor: 0.542

Best single-column rules (of 29 numeric columns):
             feature  precision@50    rank_by
    content_age_days          0.78 high-first
   scroll_events_90d          0.78 high-first
       search_volume          0.76  low-first
          word_count          0.74 high-first
engaged_sessions_90d          0.70  low-first
     engagement_rate          0.70  low-first
      age_tier_order          0.68  low-first
   sessions_last_30d          0.68  low-first

Best one-column rule: 0.780 vs floor 0.542  (lift +0.238)
-> no single threshold separates the groups cleanly; the signal lives in the interactions.

Medians of the two groups (how much they overlap):
           days_since_last_update  avg_position   ctr  impressions_90d
declining                                                             
not_down                     20.0         10.05  0.04            472.0
down                         20.0         11.30  0.08            961.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.